<div style="background: linear-gradient(135deg, #1A1226 0%, #2D1B3D 100%); padding: 28px 32px; border-radius: 12px; font-family: system-ui, sans-serif;">
<h1 style="color: #F0C040; font-size: 2.2em; margin: 0; letter-spacing: 1px;">📊 Group Project — Instructor Dashboard</h1>
<h2 style="color: #ffffff; font-size: 1.05em; margin: 8px 0 0 0; font-weight: normal;">SCTC 1013 · Elements of Data Science · Spring 2026</h2>
<p style="color: #B0A8B9; margin: 10px 0 0 0; font-size: 0.85em;">Live view of all group submissions · auto-refreshes every 20 s, or click Refresh manually.</p>
</div>


### ⚙️ Configuration

In [16]:
# Must match student notebook
SUBMISSIONS_FILE = '/home/jovyan/shared/group_project/submissions.json'

CHERRY = '#9E1B34'
GOLD   = '#F0C040'
DARK   = '#1A1226'

DOMAIN_COLORS = {
    'DS17': '#4A90D9', 'DS16': '#4A90D9', 'DS20': '#4A90D9',  # Bioinformatics
    'DS1' : '#2E7D50',                                          # Ecology
    'DS2' : '#9E1B34', 'DS9' : '#9E1B34', 'DS10': '#9E1B34',  # Public Health
    'DS3' : '#8B5CF6',                                          # Chemistry
    'DS5' : '#059669', 'DS6' : '#059669', 'DS7' : '#059669',  # Env Sci
    'DS8' : '#059669', 'DS14': '#059669', 'CAMELS': '#059669',
    'DS11': '#D97706', 'DS12': '#D97706', 'DS18': '#D97706',  # General
}

def get_color(dataset_str):
    for key, color in DOMAIN_COLORS.items():
        if key in dataset_str:
            return color
    return '#6B7280'

# Instructor creates the shared directory on first run
import os
os.makedirs(os.path.dirname(SUBMISSIONS_FILE), exist_ok=True)
print('Config loaded — reading from:', SUBMISSIONS_FILE)

Config loaded — reading from: /home/jovyan/shared/group_project/submissions.json


---
### Setup

In [2]:
import json, os, time, threading
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print('Ready')

Ready


---
### Dashboard renderer

In [3]:

def load_submissions():
    try:
        with open(SUBMISSIONS_FILE) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def render_summary_bar(subs):
    n = len(subs)
    datasets_used = len({s.get('dataset','') for s in subs})
    members_total = sum(len(s.get('members', [])) for s in subs)
    
    stats = [
        ('🏷️', str(n), 'Groups submitted'),
        ('📂', str(datasets_used), 'Unique datasets'),
        ('👥', str(members_total), 'Total members'),
    ]
    cards = ''.join(
        f'<div style="flex:1;min-width:120px;background:#F7F5F0;border-radius:10px;'
        f'padding:14px;text-align:center;">'
        f'<div style="font-size:1.4em;">{icon}</div>'
        f'<div style="font-size:2em;font-weight:800;color:#1A1226;margin:4px 0 2px;">{val}</div>'
        f'<div style="font-size:11px;color:#888;">{label}</div>'
        f'</div>'
        for icon, val, label in stats
    )
    return f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:20px;">{cards}</div>'


def render_group_card(sub):
    team   = sub.get('team', 'Unknown')
    title  = sub.get('title', '—')
    ds     = sub.get('dataset', '—')
    hyp    = sub.get('hypothesis', '')
    desc   = sub.get('description', '')
    members = sub.get('members', [])
    ts     = sub.get('submitted_at', '')
    
    color = get_color(ds)
    
    member_chips = ''.join(
        f'<span style="display:inline-block;background:#F0EDE8;border-radius:20px;'
        f'padding:3px 10px;font-size:11px;margin:2px;color:#333;">'
        f'<strong>{m["name"]}</strong> '
        f'<span style="color:#888;">· {m["role"]}</span></span>'
        for m in members
    )
    
    return f'''
    <div style="background:#fff;border:1px solid #E0DDD6;border-radius:10px;
                margin-bottom:14px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.05);">
      <!-- Card header -->
      <div style="background:{color};padding:10px 16px;display:flex;align-items:center;gap:10px;">
        <span style="font-size:1.3em;font-weight:800;color:#fff;letter-spacing:0.5px;">{team}</span>
        <span style="margin-left:auto;font-size:11px;color:rgba(255,255,255,0.75);">{ts}</span>
      </div>
      <!-- Card body -->
      <div style="padding:14px 18px;">
        <div style="font-size:15px;font-weight:700;color:#1A1226;margin-bottom:2px;">{title}</div>
        <div style="font-size:12px;color:#888;margin-bottom:10px;">📂 {ds}</div>
        
        <div style="background:#FDF8F0;border-left:3px solid {color};border-radius:4px;
                    padding:8px 12px;margin-bottom:10px;">
          <div style="font-size:10px;font-weight:700;color:{color};letter-spacing:0.5px;margin-bottom:3px;">HYPOTHESIS</div>
          <div style="font-size:12.5px;color:#333;font-style:italic;">{hyp}</div>
        </div>
        
        <div style="font-size:12px;color:#555;margin-bottom:10px;">{desc}</div>
        
        <div style="border-top:1px solid #F0EDE8;padding-top:8px;">
          <span style="font-size:11px;font-weight:600;color:#888;">TEAM:  </span>
          {member_chips}
        </div>
      </div>
    </div>
    '''


def render_dataset_coverage(subs):
    """Show which datasets are claimed so far."""
    claimed = {}
    for s in subs:
        ds = s.get('dataset', '')
        claimed[ds] = claimed.get(ds, []) + [s.get('team', '?')]
    
    if not claimed:
        return ''
    
    rows = ''.join(
        f'<tr style="border-bottom:1px solid #F0EDE8;">'
        f'<td style="padding:5px 10px;font-size:12px;color:#444;">{ds}</td>'
        f'<td style="padding:5px 10px;font-size:12px;color:#9E1B34;">' +
        ', '.join(teams) + '</td></tr>'
        for ds, teams in sorted(claimed.items())
    )
    return f'''
    <div style="background:#fff;border:1px solid #E0DDD6;border-radius:10px;
                margin-bottom:16px;overflow:hidden;">
      <div style="padding:10px 16px;border-bottom:1px solid #F0EDE8;background:#F7F5F0;">
        <strong>📂 Dataset Coverage</strong>
        <span style="font-size:11px;color:#aaa;margin-left:8px;">which groups claimed which dataset</span>
      </div>
      <table style="width:100%;border-collapse:collapse;">
        <thead><tr style="background:#F7F5F0;">
          <th style="padding:5px 10px;text-align:left;font-size:11px;color:#888;">Dataset</th>
          <th style="padding:5px 10px;text-align:left;font-size:11px;color:#888;">Group(s)</th>
        </tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
    '''


def render_all(output_widget):
    subs = load_submissions()
    ts   = time.strftime('%H:%M:%S')
    
    parts = [
        render_summary_bar(subs),
        f'<div style="font-size:11px;color:#aaa;margin-bottom:16px;">Last refreshed: {ts} · {len(subs)} group(s) submitted</div>',
    ]
    
    if subs:
        parts.append(render_dataset_coverage(subs))
        parts.append('<div style="font-size:14px;font-weight:700;color:#1A1226;margin-bottom:10px;">📋 Group Proposals</div>')
        for sub in sorted(subs, key=lambda x: x.get('timestamp', '')):
            parts.append(render_group_card(sub))
    else:
        parts.append(
            '<div style="color:#888;font-style:italic;padding:20px;text-align:center;">'
            'No submissions yet. Groups should run the <strong>GroupProject_Submission</strong> notebook.</div>'
        )
    
    with output_widget:
        clear_output(wait=True)
        display(HTML('\n'.join(parts)))

print('Renderer ready')


Renderer ready


---
### 🚀 Launch Dashboard

In [14]:

btn_refresh = widgets.Button(
    description='🔄 Refresh Now',
    layout=widgets.Layout(width='160px', height='36px'),
    style={'button_color': CHERRY, 'font_weight': 'bold'}
)
btn_toggle = widgets.Button(
    description='⏸ Pause Auto-Refresh',
    layout=widgets.Layout(width='210px', height='36px')
)
interval_slider = widgets.IntSlider(
    value=20, min=5, max=60, step=5,
    description='Interval (s):',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px')
)
status_lbl = widgets.Label(value='Auto-refresh: ON')
output     = widgets.Output()

_auto = [True]
_stop = threading.Event()

def do_refresh(_=None):
    render_all(output)

def toggle_auto(_):
    _auto[0] = not _auto[0]
    if _auto[0]:
        btn_toggle.description = '⏸ Pause Auto-Refresh'
        status_lbl.value = 'Auto-refresh: ON'
    else:
        btn_toggle.description = '▶ Resume Auto-Refresh'
        status_lbl.value = 'Auto-refresh: PAUSED'

btn_refresh.on_click(do_refresh)
btn_toggle.on_click(toggle_auto)

def _loop():
    while not _stop.is_set():
        secs = interval_slider.value
        for _ in range(secs):
            if _stop.is_set():
                return
            import time; time.sleep(1)
        if _auto[0]:
            render_all(output)

_stop.clear()
import threading
threading.Thread(target=_loop, daemon=True).start()

controls = widgets.HBox(
    [btn_refresh, btn_toggle, interval_slider, status_lbl],
    layout=widgets.Layout(gap='12px', align_items='center', margin='0 0 12px 0')
)
display(controls, output)
do_refresh()
print('Dashboard live!')


Output()

Dashboard live!


---
### ✏️ Manual Entry (instructor fallback)

In [5]:
import tempfile

def manual_submit(team, title, dataset, hypothesis, description, members):
    """
    Manual entry for groups who cannot submit from their notebook.
    members: list of dicts with 'name' and 'role' keys
    
    Example:
    manual_submit(
        team='Broad Street Bootstrappers',
        title='Microbiome Diversity in Health vs. Disease',
        dataset='DS17 · Human Microbiome Diversity',
        hypothesis='Gut microbiome Shannon diversity is lower in IBD patients vs. healthy controls.',
        description='We will use permutation tests to compare diversity indices across health groups.',
        members=[{'name': 'Alice Chen', 'role': 'Coder'}, {'name': 'Bob Kim', 'role': 'Analyst'}]
    )
    """
    entry = {
        'team': team,
        'title': title,
        'dataset': dataset,
        'hypothesis': hypothesis,
        'description': description,
        'members': members,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'submitted_at': time.strftime('%H:%M'),
        'manual': True
    }
    try:
        with open(SUBMISSIONS_FILE) as f:
            subs = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        subs = []
    subs = [s for s in subs if s.get('team', '').lower() != team.lower()]
    subs.append(entry)
    dir_ = os.path.dirname(SUBMISSIONS_FILE)
    os.makedirs(dir_, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', dir=dir_, delete=False, suffix='.tmp') as f:
        json.dump(subs, f, indent=2)
        tmp = f.name
    os.replace(tmp, SUBMISSIONS_FILE)
    print(f'✅ Submitted: {team} · {dataset}')

# Example — uncomment and edit:
# manual_submit(
#     team='Broad Street Bootstrappers',
#     title='Microbiome Diversity in Health vs. Disease',
#     dataset='DS17 · Human Microbiome Diversity',
#     hypothesis='Gut microbiome Shannon diversity is lower in IBD patients vs. healthy controls.',
#     description='We will apply permutation tests to compare diversity indices across health status groups, and bootstrap CIs for the difference in means.',
#     members=[{'name': 'Alice Chen', 'role': 'Coder'}, {'name': 'Bob Kim', 'role': 'Analyst'}]
# )


---
### 🌐 Publish to GitHub Pages
Renders submissions as a static HTML page and pushes it to your GitHub repo.
Set your PAT and repo details once, then click **Publish** any time.

In [6]:
# ── GitHub configuration ─────────────────────────────────────────
import os

# Load token from ~/.github_pat (never stored in the repo)
_pat_file = os.path.expanduser("~/.github_pat")
if os.path.exists(_pat_file):
    with open(_pat_file) as f:
        GH_TOKEN = f.read().strip()
else:
    GH_TOKEN = os.environ.get("GITHUB_PAT", "")

if not GH_TOKEN:
    raise ValueError("No GitHub token found. See setup instructions.")

GH_REPO   = 'laserchemist/data'
GH_PATH   = 'Spring 2026/Group-project/group_project_proposals.html'
GH_BRANCH = 'main'
# ─────────────────────────────────────────────────────────────────

import base64, urllib.request, urllib.error, urllib.parse

def build_html(subs):
    ts = time.strftime('%B %d, %Y at %H:%M')
    n  = len(subs)

    def domain_color(ds):
        for k, c in [('DS17','#4A90D9'),('DS16','#4A90D9'),('DS20','#4A90D9'),
                     ('DS1','#2E7D50'),('DS2','#9E1B34'),('DS9','#9E1B34'),
                     ('DS10','#9E1B34'),('DS3','#8B5CF6'),('DS5','#059669'),
                     ('DS6','#059669'),('DS7','#059669'),('DS8','#059669'),
                     ('DS14','#059669'),('CAMELS','#059669'),
                     ('DS11','#D97706'),('DS12','#D97706'),('DS18','#D97706')]:
            if k in ds: return c
        return '#6B7280'

    cards = ''
    for sub in sorted(subs, key=lambda x: x.get('team','')):
        team    = sub.get('team','')
        title   = sub.get('title','\u2014')
        ds      = sub.get('dataset','\u2014')
        hyp     = sub.get('hypothesis','')
        desc    = sub.get('description','')
        members = sub.get('members',[])
        ts_sub  = sub.get('submitted_at','')
        color   = domain_color(ds)
        chips   = ' '.join(
            f'<span style="background:#F0EDE8;border-radius:20px;padding:2px 10px;'
            f'font-size:12px;margin:2px;display:inline-block;">'
            f'<strong>{m["name"]}</strong> <span style="color:#888;">&middot; {m["role"]}</span></span>'
            for m in members
        )
        cards += f'''
        <div class="card">
          <div class="card-header" style="background:{color}">
            <span class="team-name">{team}</span>
            <span class="ts">{ts_sub}</span>
          </div>
          <div class="card-body">
            <div class="title">{title}</div>
            <div class="dataset">\U0001f4c2 {ds}</div>
            <div class="hyp-box" style="border-left:3px solid {color}">
              <div class="hyp-label" style="color:{color}">HYPOTHESIS</div>
              <div class="hyp-text">{hyp}</div>
            </div>
            <div class="desc">{desc}</div>
            <div class="members">{chips}</div>
          </div>
        </div>'''

    n_datasets = len({s.get('dataset') for s in subs})
    n_members  = sum(len(s.get('members', [])) for s in subs)
    return f'''<!DOCTYPE html>
<html lang="en"><head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Group Project Proposals \u2014 EDS Spring 2026</title>
<style>
  body{{font-family:system-ui,sans-serif;background:#F5F2EC;margin:0;padding:24px;color:#222}}
  .hdr{{background:linear-gradient(135deg,#1A1226,#2D1B3D);color:#fff;padding:28px 32px;
        border-radius:12px;border-left:6px solid #9E1B34;margin-bottom:20px}}
  .hdr h1{{color:#F0C040;margin:0;font-size:1.8em}}
  .hdr p{{color:#B0A8B9;margin:6px 0 0;font-size:.9em}}
  .stats{{display:flex;gap:12px;flex-wrap:wrap;margin-bottom:20px}}
  .stat{{background:#fff;border-radius:10px;padding:14px 20px;text-align:center;
         flex:1;min-width:100px;box-shadow:0 2px 6px rgba(0,0,0,.06)}}
  .stat-n{{font-size:2em;font-weight:800;color:#1A1226}}
  .stat-l{{font-size:11px;color:#888}}
  .card{{background:#fff;border-radius:10px;margin-bottom:14px;
          box-shadow:0 2px 8px rgba(0,0,0,.06);overflow:hidden}}
  .card-header{{padding:10px 18px;display:flex;align-items:center;justify-content:space-between}}
  .team-name{{font-size:1.1em;font-weight:800;color:#fff}}
  .ts{{font-size:11px;color:rgba(255,255,255,.7)}}
  .card-body{{padding:14px 18px}}
  .title{{font-size:15px;font-weight:700;color:#1A1226}}
  .dataset{{font-size:12px;color:#888;margin:4px 0 10px}}
  .hyp-box{{background:#FDF8F0;border-radius:4px;padding:8px 12px;margin-bottom:10px}}
  .hyp-label{{font-size:10px;font-weight:700;letter-spacing:.5px;margin-bottom:3px}}
  .hyp-text{{font-size:12.5px;color:#333;font-style:italic}}
  .desc{{font-size:12px;color:#555;margin-bottom:10px}}
  .members{{border-top:1px solid #F0EDE8;padding-top:8px}}
  footer{{text-align:center;font-size:11px;color:#aaa;margin-top:24px}}
</style></head><body>
<div class="hdr"><h1>\U0001f4cb Group Project Proposals</h1>
<p>SCTC 1013 &middot; Elements of Data Science &middot; Spring 2026 &nbsp;&middot;&nbsp; {n} group(s) &middot; Updated {ts}</p></div>
<div class="stats">
  <div class="stat"><div class="stat-n">{n}</div><div class="stat-l">Groups</div></div>
  <div class="stat"><div class="stat-n">{n_datasets}</div><div class="stat-l">Datasets</div></div>
  <div class="stat"><div class="stat-n">{n_members}</div><div class="stat-l">Members</div></div>
</div>
{cards}
<footer>Generated by EDS GroupProject_Dashboard &middot; Temple University</footer>
</body></html>'''


def push_to_github(html_content, token, repo, path, branch):
    api = f'https://api.github.com/repos/{repo}/contents/{urllib.parse.quote(path)}'
    headers = {'Authorization': f'token {token}',
               'Accept': 'application/vnd.github+json',
               'Content-Type': 'application/json'}
    sha = None
    req = urllib.request.Request(api, headers=headers)
    try:
        with urllib.request.urlopen(req) as r:
            sha = json.loads(r.read())['sha']
    except urllib.error.HTTPError as e:
        if e.code != 404: raise
    encoded = base64.b64encode(html_content.encode()).decode()
    payload = {'message': f'Update group project proposals [{time.strftime("%Y-%m-%d %H:%M")}]',
                'content': encoded, 'branch': branch}
    if sha: payload['sha'] = sha
    req = urllib.request.Request(api, data=json.dumps(payload).encode(), headers=headers, method='PUT')
    with urllib.request.urlopen(req) as r:
        result = json.loads(r.read())
    owner, repo_name = repo.split('/')
    return f'https://{owner}.github.io/{repo_name}/{path}'


btn_publish  = widgets.Button(
    description='\U0001f310 Publish to GitHub',
    layout=widgets.Layout(width='220px', height='40px'),
    style={'button_color': '#2E7D50', 'font_weight': 'bold'}
)
out_publish = widgets.Output()

def on_publish(_):
    with out_publish:
        clear_output(wait=True)
        if GH_TOKEN == 'ghp_your_token_here':
            print('\u26a0\ufe0f  Set GH_TOKEN to your GitHub PAT first.')
            return
        try:
            subs      = load_submissions()
            html      = build_html(subs)
            pages_url = push_to_github(html, GH_TOKEN, GH_REPO, GH_PATH, GH_BRANCH)
            display(HTML(f'''
            <div style="background:#F0FFF4;border:2px solid #2E7D50;border-radius:8px;
                        padding:14px 18px;font-family:system-ui;max-width:520px;">
              <div style="font-size:1.1em;">\u2705 <strong style="color:#2E7D50;">Published!</strong>
                   &nbsp;{len(subs)} group(s) &middot; {time.strftime("%H:%M")}</div>
              <div style="margin-top:8px;font-size:13px;">
                <strong>GitHub Pages URL:</strong><br>
                <a href="{pages_url}" target="_blank" style="color:#2E7D50;word-break:break-all;">{pages_url}</a>
              </div>
              <div style="margin-top:6px;font-size:11px;color:#888;">
                Pages may take ~30 s to update. Share this link with students to let them verify their submission.
              </div>
            </div>'''))
        except Exception as e:
            print(f'\u274c Error: {e}')

btn_publish.on_click(on_publish)
display(
    widgets.HTML('<div style="font-size:12px;color:#555;margin-bottom:8px;">'
                 'Requires GitHub Pages enabled on your repo '
                 '(<em>Settings \u2192 Pages \u2192 Deploy from branch: main</em>). '
                 'PAT needs <strong>repo</strong> scope.</div>'),
    btn_publish, out_publish
)

HTML(value='<div style="font-size:12px;color:#555;margin-bottom:8px;">Requires GitHub Pages enabled on your re…

Button(description='🌐 Publish to GitHub', layout=Layout(height='40px', width='220px'), style=ButtonStyle(butto…

Output()

---
### 🗑️ Reset (wipe all submissions — use with caution)

In [9]:
import os
if os.path.exists(SUBMISSIONS_FILE):
    os.remove(SUBMISSIONS_FILE)
    print('Submissions cleared.')

Submissions cleared.
